In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedShuffleSplit

In [2]:
df = pd.read_csv("../data/raw/housing.csv")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [3]:
df["income_cat"] = pd.cut(
    df["median_income"],
    bins=[0, 1.5, 3.0, 4.5, 6.0, np.inf],
    labels=[1, 2, 3, 4, 5]
)

split = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

for train_idx, test_idx in split.split(df, df["income_cat"]):
    strat_train_set = df.loc[train_idx].drop("income_cat", axis=1)
    strat_test_set = df.loc[test_idx].drop("income_cat", axis=1)

In [4]:
housing = strat_train_set.copy()

housing_labels = housing["median_house_value"].copy()

housing = housing.drop("median_house_value", axis=1)

## Feature Engineering

In [5]:
# Rooms per household
housing["rooms_per_household"] = (
    housing["total_rooms"] / housing["households"]
)

# Bedrooms per room
housing["bedrooms_per_room"] = (
    housing["total_bedrooms"] / housing["total_rooms"]
)

# Population per household
housing["population_per_household"] = (
    housing["population"] / housing["households"]
)

In [6]:
housing.shape

(16512, 12)

In [7]:
housing[["rooms_per_household", "bedrooms_per_room", "population_per_household"]].isna().sum()

rooms_per_household           0
bedrooms_per_room           158
population_per_household      0
dtype: int64

In [8]:
housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity,rooms_per_household,bedrooms_per_room,population_per_household
12655,-121.46,38.52,29.0,3873.0,797.0,2237.0,706.0,2.1736,INLAND,5.485836,0.205784,3.168555
15502,-117.23,33.09,7.0,5320.0,855.0,2015.0,768.0,6.3373,NEAR OCEAN,6.927083,0.160714,2.623698
2908,-119.04,35.37,44.0,1618.0,310.0,667.0,300.0,2.8750,INLAND,5.393333,0.191595,2.223333
14053,-117.13,32.75,24.0,1877.0,519.0,898.0,483.0,2.2264,NEAR OCEAN,3.886128,0.276505,1.859213
20496,-118.70,34.28,27.0,3536.0,646.0,1837.0,580.0,4.4964,<1H OCEAN,6.096552,0.182692,3.167241


In [9]:
housing_nums = housing.select_dtypes(include="number").columns.tolist()
housing_nums

['longitude',
 'latitude',
 'housing_median_age',
 'total_rooms',
 'total_bedrooms',
 'population',
 'households',
 'median_income',
 'rooms_per_household',
 'bedrooms_per_room',
 'population_per_household']

In [10]:
housing_cat = ["ocean_proximity"]

In [12]:
# Preprocessing pipeline

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

full_pipeline = ColumnTransformer([
    ("num", num_pipeline, housing_nums),
    ("cat", cat_pipeline, housing_cat)
])

housing_processed = full_pipeline.fit_transform(housing)

In [13]:
housing_processed.shape

(16512, 16)

## Model Training after Feature Engineering

In [15]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

# Linear Regression model
lin_reg = LinearRegression()
lin_reg.fit(housing_processed, housing_labels)
lin_preds = lin_reg.predict(housing_processed)
lin_rmse = root_mean_squared_error(housing_labels, lin_preds)
print("RMSE of Linear Regressor is:- ", lin_rmse)

# Decision Tree Regression model
dt_reg = DecisionTreeRegressor()
dt_reg.fit(housing_processed, housing_labels)
dt_preds = dt_reg.predict(housing_processed)
dt_rmse = root_mean_squared_error(housing_labels, dt_preds)
print("RMSE of Decision Tree Regressor is:- ", dt_rmse)

# Random Forest Regression model
rf_reg = RandomForestRegressor()
rf_reg.fit(housing_processed, housing_labels)
rf_preds = rf_reg.predict(housing_processed)
rf_rmse = root_mean_squared_error(housing_labels, rf_preds)
print("RMSE of Random Forest Regressor is:- ", rf_rmse)

RMSE of Linear Regressor is:-  68160.92435491859
RMSE of Decision Tree Regressor is:-  0.0
RMSE of Random Forest Regressor is:-  18630.75492091416


## Cross Validation after Feature Engineering

In [16]:
from sklearn.model_selection import cross_val_score

# Linear Regression
lin_scores = cross_val_score(
    lin_reg,
    housing_processed,
    housing_labels,
    scoring="neg_root_mean_squared_error",
    cv=5
)
lin_rmse_scores = -lin_scores

print("Linear Regression:")
print("Scores:", lin_rmse_scores)
print("Mean:", lin_rmse_scores.mean())
print("Standard deviation:", lin_rmse_scores.std())

# Decision Tree
dt_scores = cross_val_score(
    dt_reg,
    housing_processed,
    housing_labels,
    scoring="neg_root_mean_squared_error",
    cv=5
)
dt_rmse_scores = -dt_scores

print("\nDecision Tree:")
print("Scores:", dt_rmse_scores)
print("Mean:", dt_rmse_scores.mean())
print("Standard deviation:", dt_rmse_scores.std())

# Random Forest
rf_scores = cross_val_score(
    rf_reg,
    housing_processed,
    housing_labels,
    scoring="neg_root_mean_squared_error",
    cv=5
)
rf_rmse_scores = -rf_scores

print("\nRandom Forest:")
print("Scores:", rf_rmse_scores)
print("Mean:", rf_rmse_scores.mean())
print("Standard deviation:", rf_rmse_scores.std())

Linear Regression:
Scores: [67609.56917321 67668.73748845 69483.59713542 69315.39303326
 68092.25359621]
Mean: 68433.9100853107
Standard deviation: 807.5408707429865

Decision Tree:
Scores: [69373.89795876 71386.40484766 69084.44843879 72982.38292635
 71471.1245949 ]
Mean: 70859.65175329268
Standard deviation: 1450.2528494932144

Random Forest:
Scores: [50369.91722077 49998.58934608 49638.30178707 51640.02427821
 51805.46233131]
Mean: 50690.45899268895
Standard deviation: 875.599462923132
